this script has been created to better identify ethnciites in the care home cohort population, using code from colleagues at ASR (K.Best and L.Gong). it makes use of multiple data sources to identify ethnicity in inidivduals resulting in a lower missing rate

In [5]:
library(ggplot2)
library(bigrquery)
library(tidyverse)
library(lubridate)
library(stats)
library(finalfit)
ProjectId = "yhcr-prd-bradfor-bia-core"

targetdb2 <- 'yhcr-prd-bradfor-bia-core.CB_FDM_DeathCertificates'
targetdb2 <-gsub(' ','',targetdb2)

targetdb3 <- 'yhcr-prd-bradfor-bia-core.CB_2172'
targetdb3 <- gsub(' ','',targetdb3)

targetdb4 <- 'yhcr-prd-bradfor-bia-core.CB_FDM_PrimaryCare'
targetdb4 <- gsub(' ','',targetdb4)



get the ethnciites used in primary care data person table

In [26]:
sql1 <- 
paste(
    '
with ethnicity_prep as (SELECT 
person_id,
ethnicity,
from 
',targetdb4,'.tbl_srpatient
where
ethnicity !="Ethnic category - 2011 census England and Wales"  AND

ethnicity !="O/E - ethnic group NOS"  AND

ethnicity !="O/E - ethnic group"  AND

ethnicity !="Ethnic groups (census) NOS"  AND

ethnicity !="Not specified - JDB"  AND

person_id in( select person_id from ',targetdb3,'.care_home_cohort_v1) 
ORDER BY PERSON_ID),

sr_ethnicity as (
SELECT person_id , ethnicity,

FROM (

    SELECT ROW_NUMBER() OVER (PARTITION BY person_id ORDER BY RAND()) AS row_num,

        person_id, ethnicity

    FROM ethnicity_prep

) AS T
WHERE row_num = 1),

ethnicity_join as (
SELECT

a.person_id,

a.gender_source_value as gender,

  a.age_admission,

  b.ethnicity_concept_id,


  d.Ethnicity as Ethnicity_SR_Table,

FROM ',targetdb3,'.care_home_cohort_v1 as a

LEFT JOIN  ',targetdb4,'.person as b

on a.person_id=b.person_id

Left JOIN sr_ethnicity as d

on a.person_id=d.person_id)

select
*
from 
ethnicity_join

;',sep="")



bp_tbl <-bq_project_query(ProjectId,sql1)
bp_data <- bq_table_download(bp_tbl)

In [ ]:
bp_data